In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import os
access_token = os.getenv('HUGGINGFACE_TOKEN')
# default: Load the model on the available device(s)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", torch_dtype="auto", attn_implementation="flash_attention_2",device_map="auto",token=access_token
)




processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", token=access_token)

# The default range for the number of visual tokens per image in the model is 4-16384.
# You can set min_pixels and max_pixels according to your needs, such as a token range of 256-1280, to balance performance and cost.
# min_pixels = 256*28*28
# max_pixels = 1280*28*28
# processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels)

messages = [
    {"role": "system", "content": "You are an expert in urban ecology, botany, and behavioral sociology. Your task is to analyze images from urban parks to extract structured data regarding biodiversity and human-nature interactions."},
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "file:///home/dream/Study/26p1/DSproject/shenzhen/6深圳_携程图像文本/5深圳市罗湖区仙湖植物园/class_1/5深圳市罗湖区仙湖植物园_233539_2.jpg"},
            {"type": "image", "image": "file:///home/dream/Study/26p1/DSproject/shenzhen/6深圳_携程图像文本/5深圳市罗湖区仙湖植物园/class_1/5深圳市罗湖区仙湖植物园_233539_9.jpg"},
            {"type": "image", "image": "file:///home/dream/Study/26p1/DSproject/shenzhen/6深圳_携程图像文本/5深圳市罗湖区仙湖植物园/class_1/5深圳市罗湖区仙湖植物园_233539_10.jpg"},
            {"type": "image", "image": "file:///home/dream/Study/26p1/DSproject/shenzhen/6深圳_携程图像文本/5深圳市罗湖区仙湖植物园/class_1/5深圳市罗湖区仙湖植物园_233539_14.jpg"},
            {"type": "text", "text": """Task 1: Fine-grained Species Identification Identify all distinct plant and animal species visible in the image. Precision: You must provide the specific Species level name (e.g., "Tulip", "Lotus", "Mallard", "Spotted Deer"). Do NOT use generic labels like "Flower", "Bird", or "Plant" unless the image resolution makes specific identification impossible. Task 2: Context-aware Activity Recognition Analyze human behavior if people are present. Activity: Identify the specific leisure or recreational activity (e.g., "Camping", "Running", "Photography", "Walking"). Social Mode: Determine the social interaction mode. Classify as either "Solitary" (one person) or "Group" (two or more people interacting). 
Output Format: Return the analysis strictly as a valid JSON object with no additional markdown or conversational text. Use the following schema:
{
  "biodiversity": [{
      "species_label": "String (e.g., 'Nelumbo nucifera' or 'Lotus')",
      "category": "String (e.g., 'Plant' or 'Animal')",}],
  "human_activity": [{
      "activity_label": "String (e.g., 'Camping')",
      "social_mode": "String ('Solitary' or 'Group')"}]
}"""},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

/home/dream/anaconda3/envs/dsproject/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 729/729 [00:01<00:00, 687.64it/s, Materializing param=model.visual.patch_embed.proj.weight]                          
Some parameters are on the meta device because they were offloaded to the cpu.
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


['```json\n{\n  "biodiversity": [\n    {\n      "species_label": "Delphinium",\n      "category": "Plant"\n    },\n    {\n      "species_label": "Topiary",\n      "category": "Plant"\n    },\n    {\n      "species_label": "Azalea",\n      "category": "Plant"\n    },\n    {\n      "species_label": "Butterfly",\n      "category": "Animal"\n    }\n  ],\n  "human_activity": []\n}\n```']


In [ ]:
# Sample messages for batch inference
messages1 = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "file:///path/to/image1.jpg"},
            {"type": "image", "image": "file:///path/to/image2.jpg"},
            {"type": "text", "text": "What are the common elements in these pictures?"},
        ],
    }
]
messages2 = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Who are you?"},
]
# Combine messages for batch processing
messages = [messages1, messages2]

# Preparation for batch inference
texts = [
    processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    for msg in messages
]
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=texts,
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Batch Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_texts = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_texts)
